# Walk de DATA_DIR

Recorre `DATA_DIR` con `os.walk` para inspeccionar la estructura de carpetas y contar imágenes por clase.

In [7]:
import os

DATA_DIR = os.path.join("..", "..", "..", "..", "..", "Downloads", "archive")
print("DATA_DIR:", DATA_DIR)
print("Existe:", os.path.isdir(DATA_DIR))

DATA_DIR: ..\..\..\..\..\Downloads\archive
Existe: True


## Estructura de carpetas

In [8]:
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.gif', '.bmp')

def print_tree(root, max_depth=3):
    root = os.path.normpath(root)
    root_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep) - root_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = '    ' * depth
        nombre = os.path.basename(dirpath) or dirpath
        n_imgs = sum(1 for f in filenames if f.lower().endswith(IMG_EXTS))
        print(f"{indent}{nombre}/  ({n_imgs} imgs, {len(dirnames)} subcarpetas)")

print_tree(DATA_DIR)

archive/  (0 imgs, 1 subcarpetas)
    PlantVillage/  (0 imgs, 16 subcarpetas)
        Pepper__bell___Bacterial_spot/  (997 imgs, 0 subcarpetas)
        Pepper__bell___healthy/  (1478 imgs, 0 subcarpetas)
        PlantVillage/  (0 imgs, 15 subcarpetas)
            Pepper__bell___Bacterial_spot/  (997 imgs, 0 subcarpetas)
            Pepper__bell___healthy/  (1478 imgs, 0 subcarpetas)
            Potato___Early_blight/  (1000 imgs, 0 subcarpetas)
            Potato___healthy/  (152 imgs, 0 subcarpetas)
            Potato___Late_blight/  (1000 imgs, 0 subcarpetas)
            Tomato_Bacterial_spot/  (2127 imgs, 0 subcarpetas)
            Tomato_Early_blight/  (1000 imgs, 0 subcarpetas)
            Tomato_healthy/  (1591 imgs, 0 subcarpetas)
            Tomato_Late_blight/  (1909 imgs, 0 subcarpetas)
            Tomato_Leaf_Mold/  (952 imgs, 0 subcarpetas)
            Tomato_Septoria_leaf_spot/  (1771 imgs, 0 subcarpetas)
            Tomato_Spider_mites_Two_spotted_spider_mite/  (1676 imgs

## Todas las rutas de imágenes

In [9]:
def get_image_paths(directory):
    image_paths = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(IMG_EXTS):
                image_paths.append(os.path.join(root, file))
    return image_paths

all_image_paths = get_image_paths(DATA_DIR)
print(f"Total de imágenes encontradas: {len(all_image_paths)}")
all_image_paths[:5]

Total de imágenes encontradas: 41276


['..\\..\\..\\..\\..\\Downloads\\archive\\PlantVillage\\Pepper__bell___Bacterial_spot\\0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Spot 8964.JPG',
 '..\\..\\..\\..\\..\\Downloads\\archive\\PlantVillage\\Pepper__bell___Bacterial_spot\\006adb74-934f-448f-a14f-62181742127b___JR_B.Spot 3395.JPG',
 '..\\..\\..\\..\\..\\Downloads\\archive\\PlantVillage\\Pepper__bell___Bacterial_spot\\00f2e69a-1e56-412d-8a79-fdce794a17e4___JR_B.Spot 3132.JPG',
 '..\\..\\..\\..\\..\\Downloads\\archive\\PlantVillage\\Pepper__bell___Bacterial_spot\\01613cd0-d3cd-4e96-945c-a312002037bf___JR_B.Spot 3262.JPG',
 '..\\..\\..\\..\\..\\Downloads\\archive\\PlantVillage\\Pepper__bell___Bacterial_spot\\0169b9ac-07b9-4be1-8b85-da94481f05a4___NREC_B.Spot 9169.JPG']

## Conteo de imágenes por clase (carpeta inmediata)

In [10]:
from collections import Counter
import pandas as pd

PLANT_DIR = os.path.join(DATA_DIR, "PlantVillage")  # copia 1: clases directamente aquí

def get_class_dirs(data_dir):
    return sorted(
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d)) and d.lower() != "plantvillage"
    )

conteo = Counter()
for clase in get_class_dirs(PLANT_DIR):
    cdir = os.path.join(PLANT_DIR, clase)
    n = sum(1 for f in os.listdir(cdir) if f.lower().endswith(IMG_EXTS))
    conteo[clase] = n

df_conteo = pd.DataFrame(conteo.items(), columns=["clase", "n_imagenes"]).sort_values("n_imagenes", ascending=False)
df_conteo

,clase,n_imagenes
12,Tomato__Tomato_YellowLeaf__Curl_Virus,3208
5,Tomato_Bacterial_spot,2127
7,Tomato_Late_blight,1909
9,Tomato_Septoria_leaf_spot,1771
10,Tomato_Spider_mites_Two_spotted_spider_mite,1676
14,Tomato_healthy,1591
1,Pepper__bell___healthy,1478
11,Tomato__Target_Spot,1404
2,Potato___Early_blight,1000
3,Potato___Late_blight,1000


## Comparar `PlantVillage/` (copia 1) vs `PlantVillage/PlantVillage/` (copia 2)

Verifica si ambas copias tienen las mismas clases, la misma cantidad de imágenes por clase, los mismos nombres de archivo y (opcional) el mismo contenido byte a byte.

In [11]:
COPIA_1 = PLANT_DIR                                  # DATA_DIR/PlantVillage
COPIA_2 = os.path.join(PLANT_DIR, "PlantVillage")     # DATA_DIR/PlantVillage/PlantVillage

def class_file_map(base_dir):
    """{clase: set(nombres_de_archivo)} para las carpetas de clase directas de base_dir."""
    mapa = {}
    for clase in get_class_dirs(base_dir):
        cdir = os.path.join(base_dir, clase)
        mapa[clase] = {f for f in os.listdir(cdir) if f.lower().endswith(IMG_EXTS)}
    return mapa

mapa_1 = class_file_map(COPIA_1)
mapa_2 = class_file_map(COPIA_2)

clases_1, clases_2 = set(mapa_1), set(mapa_2)
print("Mismas clases:", clases_1 == clases_2)
if clases_1 != clases_2:
    print("  Solo en copia 1:", clases_1 - clases_2)
    print("  Solo en copia 2:", clases_2 - clases_1)

filas = []
for clase in sorted(clases_1 & clases_2):
    f1, f2 = mapa_1[clase], mapa_2[clase]
    filas.append({
        "clase": clase,
        "n_copia1": len(f1),
        "n_copia2": len(f2),
        "mismos_nombres": f1 == f2,
        "solo_copia1": len(f1 - f2),
        "solo_copia2": len(f2 - f1),
    })

df_cmp = pd.DataFrame(filas)
print("\n¿Todas las clases idénticas (mismo set de nombres)?", df_cmp["mismos_nombres"].all())
df_cmp

Mismas clases: True

¿Todas las clases idénticas (mismo set de nombres)? True


,clase,n_copia1,n_copia2,mismos_nombres,solo_copia1,solo_copia2
0,Pepper__bell___Bacterial_spot,997,997,True,0,0
1,Pepper__bell___healthy,1478,1478,True,0,0
2,Potato___Early_blight,1000,1000,True,0,0
3,Potato___Late_blight,1000,1000,True,0,0
4,Potato___healthy,152,152,True,0,0
5,Tomato_Bacterial_spot,2127,2127,True,0,0
6,Tomato_Early_blight,1000,1000,True,0,0
7,Tomato_Late_blight,1909,1909,True,0,0
8,Tomato_Leaf_Mold,952,952,True,0,0
9,Tomato_Septoria_leaf_spot,1771,1771,True,0,0


In [12]:
import hashlib
import random

def md5_file(path, chunk_size=1 << 16):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

random.seed(42)
N_MUESTRA = 400  # imágenes por clase a hashear

filas_hash = []
for clase in sorted(clases_1 & clases_2):
    comunes = sorted(mapa_1[clase] & mapa_2[clase])
    muestra_clase = random.sample(comunes, min(N_MUESTRA, len(comunes)))
    n_diff = 0
    for nombre in muestra_clase:
        h1 = md5_file(os.path.join(COPIA_1, clase, nombre))
        h2 = md5_file(os.path.join(COPIA_2, clase, nombre))
        if h1 != h2:
            n_diff += 1
    filas_hash.append({
        "clase": clase,
        "n_muestreadas": len(muestra_clase),
        "n_diferentes": n_diff,
        "contenido_identico": n_diff == 0,
    })

df_hash = pd.DataFrame(filas_hash)
print("¿Contenido idéntico en toda la muestra?", df_hash["contenido_identico"].all())
df_hash

¿Contenido idéntico en toda la muestra? True


,clase,n_muestreadas,n_diferentes,contenido_identico
0,Pepper__bell___Bacterial_spot,400,0,True
1,Pepper__bell___healthy,400,0,True
2,Potato___Early_blight,400,0,True
3,Potato___Late_blight,400,0,True
4,Potato___healthy,152,0,True
5,Tomato_Bacterial_spot,400,0,True
6,Tomato_Early_blight,400,0,True
7,Tomato_Late_blight,400,0,True
8,Tomato_Leaf_Mold,400,0,True
9,Tomato_Septoria_leaf_spot,400,0,True
